In [1]:
!pip install -q pymupdf easyocr layoutparser torch torchvision
!pip install -q "detectron2@git+https://github.com/facebookresearch/detectron2.git@v0.6"
!pip install -q "git+https://github.com/Layout-Parser/layout-parser.git"


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> [350 lines of output]
      running bdist_wheel
      running build
      running build_py
      creating build
      creating build\lib.win-amd64-cpython-311
      creating build\lib.win-amd64-cpython-311\detectron2
      copying detectron2\__init__.py -> build\lib.win-amd64-cpython-311\detectron2
      creating build\lib.win-amd64-cpython-311\tools
      copying tools\analyze_model.py -> build\lib.win-amd64-cpython-311\tools
      copying tools\benchmark.py -> build\lib.win-amd64-cpython-311\tools
      copying tools\convert-torchvision-to-d2.py -> build\lib.win-amd64-cpython-311\tools
      copying tools\lazyconfig_train_net.py -> build\lib.win-amd64-cpython-311\tools
      copying tools\lightning_train_net.py -> build\lib.win-amd64-cpython

^C



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# Libraries

In [ ]:
import os
import fitz  # PyMuPDF
import cv2
import easyocr
import re
import requests
from urllib.parse import urlparse
import layoutparser as lp
import numpy as np
from PIL import Image

# Initialize EasyOCR reader
reader = easyocr.Reader(['vi'], gpu=True)

try:
    layout_model = lp.Detectron2LayoutModel(
        'lp://PubLayNet/faster_rcnn_R_50_FPN_3x/config',
        extra_config=["MODEL.ROI_HEADS.SCORE_THRESH_TEST", 0.5],
        label_map={0: "Text", 1: "Title", 2: "List", 3: "Table", 4: "Figure"}
    )
except Exception as e:
    print(f"Layout model failed to load: {e}")
    layout_model = None

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% CompleteLayout model failed to load: module layoutparser has no attribute Detectron2LayoutModel


# Utils

In [ ]:
def download_pdf_from_url(url, save_dir="input", chunk_size=8192):
    os.makedirs(save_dir, exist_ok=True)
    filename = os.path.basename(urlparse(url).path)
    if not filename.endswith(".pdf"):
        filename = "document.pdf"

    save_path = os.path.join(save_dir, filename)
    if os.path.exists(save_path):
        print(f"PDF already exists: {save_path}")
        return save_path

    print(f"Download PDF...")
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(save_path, "wb") as f:
            for chunk in r.iter_content(chunk_size):
                if chunk:
                    f.write(chunk)

    print(f"Saved: {save_path}")
    return save_path

In [ ]:
def pdf_to_images(pdf_path, out_dir="pdf_pages", dpi=200):
    os.makedirs(out_dir, exist_ok=True)
    doc = fitz.open(pdf_path)

    image_paths = []
    for i, page in enumerate(doc):
        pix = page.get_pixmap(dpi=dpi)
        img_path = f"{out_dir}/page_{i+1:03d}.png"
        pix.save(img_path)
        image_paths.append(img_path)

    print(f"Converted {len(image_paths)} pages to images")
    return image_paths

In [ ]:
def detect_layout_blocks(image_path):
    if layout_model is None:
        return None, None

    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    layout = layout_model.detect(image_rgb)

    blocks = sorted(layout, key=lambda b: (b.coordinates[1], b.coordinates[0]))

    return blocks, image_rgb

def ocr_block(image, block):
    x1, y1, x2, y2 = map(int, block.coordinates)

    padding = 10
    x1 = max(0, x1 - padding)
    y1 = max(0, y1 - padding)
    x2 = min(image.shape[1], x2 + padding)
    y2 = min(image.shape[0], y2 + padding)

    block_img = image[y1:y2, x1:x2]

    results = reader.readtext(block_img, paragraph=False)

    # Join text preserving some structure
    texts = [text for (bbox, text, conf) in results]
    return ' '.join(texts)

def ocr_page_with_layout(image_path):
    blocks, image = detect_layout_blocks(image_path)

    # Fallback to basic OCR if layout detection fails
    if blocks is None:
        return ocr_page_text_basic(image_path)

    structured_content = []

    for block in blocks:
        block_type = block.type

        # Skip figures from text extraction
        if block_type == 'Figure':
            continue

        block_text = ocr_block(image, block)

        if block_text.strip():
            structured_content.append({
                'type': block_type,
                'text': block_text.strip(),
                'coordinates': block.coordinates
            })

    # Combine text blocks with proper separators
    full_text = []
    prev_type = None

    for item in structured_content:
        text = item['text']
        block_type = item['type']

        # Add separators based on block type
        if block_type in ['Title', 'List']:
            full_text.append('\n\n' + text + '\n\n')
        else:
            # Text blocks: separate if type changes
            if prev_type and prev_type != block_type:
                full_text.append('\n' + text + ' ')
            else:
                full_text.append(text + ' ')

        prev_type = block_type

    return ''.join(full_text)

def ocr_page_text_basic(image_path):
    results = reader.readtext(
        image_path,
        paragraph=False,
        width_ths=0.3,
        height_ths=0.3,
        ycenter_ths=0.3,
        text_threshold=0.6,
        low_text=0.3,
        link_threshold=0.3
    )

    if not results:
        return ""

    # Sort by Y position
    results_sorted = sorted(results, key=lambda x: x[0][0][1])

    text_blocks = []
    prev_y = None

    for bbox, text, conf in results_sorted:
        curr_y = bbox[0][1]

        if prev_y is not None and abs(curr_y - prev_y) > 20:
            text_blocks.append('\n')

        text_blocks.append(text)
        prev_y = curr_y

    full_text = ' '.join(text_blocks)
    full_text = re.sub(r'[ \t]+', ' ', full_text)
    full_text = re.sub(r'\n\s+\n', '\n\n', full_text)

    return full_text

In [ ]:
PDF_URL = "https://84864e12bc.vws.vegacdn.vn//data/doc/2025/thcslienninh/2025_2/26/sach-bai-tap-toan-8-tap-1-ket-noi-tri-thuc-voi-cuoc-song_262202515.pdf"

pdf_file_path = download_pdf_from_url(PDF_URL, save_dir="./input")
page_images = pdf_to_images(pdf_file_path, out_dir="output/pages", dpi=200)

Download PDF...
Saved: ./input/sach-bai-tap-toan-8-tap-1-ket-noi-tri-thuc-voi-cuoc-song_262202515.pdf
Converted 113 pages to images


In [ ]:
def extract_problems_with_figures(text):
    """
    Extract problems that mention figures from OCR text
    Handles two formats:
    1. Numbered problems: 2.2., 3.12., etc.
    2. Example problems: Ví dụ 1, Ví dụ 2, etc.

    Strategy: Text from one marker to the next marker = one complete problem
    IMPORTANT: Flexible pattern to handle OCR errors (missing dots/spaces)
    FILTER: Skip problems related to charts/graphs (biểu đồ)
    """
    figure_patterns = [
        (r'Hình\s*\d+\.\d+', 'Hình X.Y'),
        (r'\(H\.\d+\.\d+\)', '(H.X.Y)'),
        (r'trong\s+(?:các\s+)?hình\s+(?:vẽ\s+)?(?:sau|trên|dưới|bên)', 'trong hình...'),
    ]

    # Keywords to filter out chart/graph problems
    chart_keywords = [
        r'biểu\s*đồ',
        r'đồ\s*thị',
        r'chart',
        r'graph',
        r'bảng\s*thống\s*kê',
    ]

    results = []
    all_markers = []

    # Find numbered problems with FLEXIBLE pattern
    # Matches: "2.12. ", "2.12.", "2.12 ", or just "2.12"
    # But must be after line break OR after 2+ spaces (to avoid matching numbers in text)
    for match in re.finditer(r'(?:^|\n|(?<=\s{2}))(\d+\.\d+)\.?\s*', text, re.MULTILINE):
        problem_num = match.group(1)

        # Skip if it looks like a decimal in the middle of text
        # (e.g., "là 3.14 trong công thức")
        if match.start() > 0:
            before = text[max(0, match.start()-5):match.start()]
            # Skip if preceded by lowercase letters or certain words
            if re.search(r'[a-zđáàảãạăắằẳẵặâấầẩẫậéèẻẽẹêếềểễệíìỉĩịóòỏõọôốồổỗộơớờởỡợúùủũụưứừửữựýỳỷỹỵ]\s*$', before, re.IGNORECASE):
                continue

        all_markers.append({
            'type': 'numbered',
            'identifier': problem_num,
            'start': match.start(),
            'end': match.end(),
            'full_match': match.group(0).strip()
        })

    # Find example problems: Ví dụ 1, Ví dụ 2, etc.
    for match in re.finditer(r'Ví\s+dụ\s+(\d+)', text, re.IGNORECASE):
        all_markers.append({
            'type': 'example',
            'identifier': f'VD{match.group(1)}',
            'start': match.start(),
            'end': match.end(),
            'full_match': match.group(0)
        })

    # Sort all markers by position
    all_markers.sort(key=lambda x: x['start'])

    # Remove duplicate markers at same position (keep first)
    unique_markers = []
    last_pos = -100
    for marker in all_markers:
        if marker['start'] - last_pos > 5:  # At least 5 chars apart
            unique_markers.append(marker)
            last_pos = marker['start']

    all_markers = unique_markers

    if not all_markers:
        return results

    # Process each problem segment
    for i, marker in enumerate(all_markers):
        # Extract text from this marker to the next marker
        start_pos = marker['end']
        end_pos = all_markers[i+1]['start'] if i+1 < len(all_markers) else len(text)

        segment = text[start_pos:end_pos].strip()

        # Skip if segment is too short
        if len(segment) < 20:
            continue

        # FILTER: Skip if segment mentions charts/graphs
        is_chart_problem = False
        for chart_pattern in chart_keywords:
            if re.search(chart_pattern, segment, re.IGNORECASE):
                is_chart_problem = True
                break

        if is_chart_problem:
            continue

        # Find all figures with their positions
        found_figures = []
        figure_positions = []

        for fig_pattern, _ in figure_patterns:
            for match in re.finditer(fig_pattern, segment, re.IGNORECASE):
                found_figures.append(match.group(0))
                figure_positions.append(match.start())

        # Skip if no figures found
        if not found_figures:
            continue

        # Check if next marker appears inside this segment
        # (OCR might have merged problems even with flexible pattern)
        if i+1 < len(all_markers):
            next_marker_pattern = all_markers[i+1]['identifier']
            # Look for the next problem number inside segment (even without dot)
            next_match = re.search(rf'\b{re.escape(next_marker_pattern)}\b', segment)

            if next_match:
                next_pos = next_match.start()

                # Check if figures appear before the next marker
                valid_figures = []
                for fig, pos in zip(found_figures, figure_positions):
                    if pos < next_pos:
                        valid_figures.append(fig)

                # If no valid figures before next marker, skip
                if not valid_figures:
                    continue

                found_figures = valid_figures
                # Truncate segment at next marker
                segment = segment[:next_pos].strip()

        # Split question and solution by "Giải"
        giai_match = re.search(r'\bGiải\b', segment, re.IGNORECASE)

        if giai_match:
            question_part = segment[:giai_match.start()].strip()
            solution_part = segment[giai_match.end():].strip()
            has_solution = True
        else:
            question_part = segment
            solution_part = ""
            has_solution = False

        # Clean up text
        question_part = re.sub(r'^[-@•()\s=:;]+', '', question_part).strip()
        solution_part = re.sub(r'^[-@•()\s=:;]+', '', solution_part).strip()

        # Skip if too short
        if len(question_part) < 30:
            continue

        # Normalize whitespace
        question_part = re.sub(r'\s+', ' ', question_part)

        results.append({
            'problem_number': marker['identifier'],
            'problem_type': marker['type'],
            'content': segment,
            'question': question_part,
            'solution': solution_part,
            'figures': list(set(found_figures)),
            'has_solution': has_solution
        })

    return results

# Starting

In [ ]:
pages_dir = os.path.join("output", "pages")

if not os.path.exists(pages_dir):
    print(f"Not found: {pages_dir}")
else:
    # Get all page image files
    page_images = sorted([
        os.path.join(pages_dir, f)
        for f in os.listdir(pages_dir)
        if f.endswith('.png')
    ])

    use_layout = layout_model is not None
    method = "Layout Detection + OCR" if use_layout else "Basic OCR"

    print(f"Processing pages with {method}")
    print(f"Folder: {pages_dir}")
    print(f"Total pages: {len(page_images)}\n")

    # OCR each page
    page_texts = {}
    for idx, page_img in enumerate(page_images):
        page_num = idx + 1
        print(f"[{page_num}/{len(page_images)}] Processing {os.path.basename(page_img)}...", end=" ")

        if use_layout:
            text = ocr_page_with_layout(page_img)
        else:
            text = ocr_page_text_basic(page_img)

        page_texts[page_num] = text
        print(f"({len(text)} chars)")

    # Extract questions with figure references
    print("Extracting questions with figures...")

    all_problems = []
    for page_num, text in page_texts.items():
        problems = extract_problems_with_figures(text)
        if problems:
            print(f"Page {page_num}: {len(problems)} question(s)")
        for prob in problems:
            prob['page_number'] = page_num
            all_problems.append(prob)

    print(f"Total: {len(all_problems)} questions with diagrams")
    print(f"Processed {len(page_texts)} pages")

Processing pages with Basic OCR
Folder: output/pages
Total pages: 113

[1/113] Processing page_001.png... (232 chars)
[2/113] Processing page_002.png... (1 chars)
[3/113] Processing page_003.png... (1744 chars)
[4/113] Processing page_004.png... (1231 chars)
[5/113] Processing page_005.png... (1179 chars)
[6/113] Processing page_006.png... (1258 chars)
[7/113] Processing page_007.png... (1320 chars)
[8/113] Processing page_008.png... (1057 chars)
[9/113] Processing page_009.png... (1143 chars)
[10/113] Processing page_010.png... (1026 chars)
[11/113] Processing page_011.png... (526 chars)
[12/113] Processing page_012.png... (949 chars)
[13/113] Processing page_013.png... (1125 chars)
[14/113] Processing page_014.png... (574 chars)
[15/113] Processing page_015.png... (1196 chars)
[16/113] Processing page_016.png... (1279 chars)
[17/113] Processing page_017.png... (1346 chars)
[18/113] Processing page_018.png... (1492 chars)
[19/113] Processing page_019.png... (858 chars)
[20/113] Proces

# Save

In [ ]:
print(f"Total problems extracted: {len(all_problems)}")

for i, prob in enumerate(all_problems[:10], 1):
    print(f"\n{'='*80}")
    print(f"Số bài: {prob['problem_number']}")
    print(f"Trang: {prob['page_number']}")
    print(f"Hình: {', '.join(prob['figures'])}")
    print(f"Đề bài:")
    print(prob['question'])

if len(all_problems) > 10:
    print(f"\n... and {len(all_problems) - 10} more problems")

Total problems extracted: 12

Số bài: 2.12
Trang: 24
Hình: (H.2.3), Hình 2.3
Đề bài:
Từ một khối lập phương có độ dài cạnh là X + 3 (cm), ta cắt bỏ một khối Iập phương có độ dài X - 1(cm) (H.2.3). Tính thể tích còn lại; viết kết phần quả dưới dạng đa thức. X + 3 'x = 1 Hình 2.3 24

Số bài: 2.23
Trang: 30
Hình: Hình 2.4, (H.2.4)
Đề bài:
Phân tích các đa thức sau thành nhân tử: b) x2 + 7x + 6. 3X + 2; a) 2 42 miéng bia có dạng hinh tròn (H.2.4) với bán kính R(cm), người ta 2.24. Từ một có bán kính r (cm)r < R. khoét một hình tròn ở giữa miếng 5 công 7 a) Viết còn lại của phần thức tính diện tích bìa. I miếng bia biết tổng còn lại của phần b) Tính diện tích hai bán kính là 10cm và hiệu hai bán kính là 3 cm. R Hình 2.4 30

Số bài: 3.11
Trang: 90
Hình: Hình 3.10, Hình 3.9, (H.3.9)
Đề bài:
H.3.9) Do CA là tia phân giác của B A góc C nên tam giác ABC cân tại B. c = 20. Vì ABCD Đặt BAC ơ thì = Ià hinh thang cân nên Ẽ= 2a. ADC vuông tại giác M Tam nên A ADC Hình 3.9 2a + % = 90 , suy ra ACD + =

In [ ]:
import json
from datetime import datetime

output_folder = os.path.join("..", "dataset", "output", "mapped_result")
os.makedirs(output_folder, exist_ok=True)

# Prepare data structure
output_data = {
    "metadata": {
        "pdf_path": pdf_file_path,
        "total_pages": len(page_texts),
        "total_problems": len(all_problems),
        "problems_with_solution": sum(1 for p in all_problems if p.get('has_solution', False)),
        "problems_without_solution": sum(1 for p in all_problems if not p.get('has_solution', False)),
        "processed_at": datetime.now().isoformat()
    },
    "problems": []
}

for i, prob in enumerate(all_problems, 1):
    output_data["problems"].append({
        "id": i,
        "problem_number": prob['problem_number'],
        "page_number": prob.get('page_number'),
        "question": prob['question'],
        "solution": prob.get('solution', ''),
        "figure_references": prob['figures'],
        "has_solution": prob.get('has_solution', False)
    })

output_file = os.path.join(output_folder, "problems_with_figures.json")
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

print(f"Folder: {output_folder}")
print(f"File: problems_with_figures.json")
print(f"   Total problems: {output_data['metadata']['total_problems']}")
print(f"   ├─ With solution: {output_data['metadata']['problems_with_solution']}")
print(f"   └─ Without solution: {output_data['metadata']['problems_without_solution']}")
print(f"   Total pages: {output_data['metadata']['total_pages']}")

Folder: ..\dataset\output\mapped_result
File: problems_with_figures.json
   Total problems: 112
   ├─ With solution: 14
   └─ Without solution: 98
   Total pages: 113
